# Analisis Nilai 0 pada G1, G2, G3 (Student Performance: Math & Portuguese)

Notebook ini menganalisis nilai 0 pada dataset *Student Performance* (Matematika dan Bahasa Portugis): pola kemunculannya, fitur yang berasosiasi dengannya, dan sensitivitas korelasi absences–G3 terhadap keberadaannya.

| Bagian | Isi |
|---|---|
| 0–2 | Setup, load data, feature engineering |
| 3 | **RG1** Pola nilai 0 pada G1, G2, G3 |
| 4 | **RG2** Fitur yang berasosiasi dengan nilai 0 |
| 5 | **RG3** Sensitivitas korelasi absences vs G3 |
| 6 | Pengecekan angka untuk slide |

**Definisi:** `case1` = **Type A** (G2 = G3 = 0), `case2` = **Type B** (G3 = 0 dan G2 > 0).

# 0. Setup & Konfigurasi

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import PathPatch, Rectangle, Patch
from matplotlib.path import Path
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore', category=FutureWarning)

IN_COLAB = 'google.colab' in sys.modules

# ---------------- KONFIGURASI (ubah di sini) ----------------
if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/kapsel/dataset'   # <-- folder dataset di Drive
    OUT_DIR  = '/content/drive/MyDrive/kapsel/output'    # <-- folder untuk menyimpan gambar/CSV
else:
    DATA_DIR = os.environ.get('DATA_DIR', '.')           # <-- folder dataset (lokal)
    OUT_DIR  = os.environ.get('OUT_DIR', './output')     # <-- folder output (lokal)

PASS_MARK = 10      # ambang lulus
SEED      = 42      # seed bootstrap (jaga tetap agar CI sama di semua slide)
N_BOOT    = 2000    # jumlah resample bootstrap
DPI       = 220
# -------------------------------------------------------------

# 1. Load Dataset

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
def find_file(folder, *names):
    """Ambil file pertama yang ada (mendukung mat.csv maupun student-mat.csv)."""
    for n in names:
        p = os.path.join(folder, n)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"Tidak ada {names} di folder: {folder}")

# Lokasi dataset
file_path1 = find_file(DATA_DIR, 'mat.csv', 'student-mat.csv')
file_path2 = find_file(DATA_DIR, 'por.csv', 'student-por.csv')

# Load kedua dataset dengan separator titik koma
df_mat = pd.read_csv(file_path1, sep=';')
df_por = pd.read_csv(file_path2, sep=';')

# Pastikan kolom teks bertipe object (dibutuhkan untuk uji fitur kategorikal di versi pandas terbaru)
for _df in (df_mat, df_por):
    for _c in _df.select_dtypes(exclude='number').columns:
        _df[_c] = _df[_c].astype('object')

os.makedirs(OUT_DIR, exist_ok=True)

print("Dataset Matematika:", df_mat.shape)
print("Dataset Bahasa Portugis:", df_por.shape)

In [ ]:
df_por

In [ ]:
df_mat

## Eksplorasi awal dataset Matematika

In [ ]:
# Melihat 5 data pertama
display(df_mat.head())

# Melihat ukuran dataset (baris, kolom)
print("Ukuran dataset:", df_mat.shape)

# Melihat nama-nama kolom
print("Nama kolom:")
print(df_mat.columns.tolist())

# Melihat tipe data dan jumlah data yang terisi
df_mat.info()

# 2. Feature Engineering

### 2.1 Status kelulusan serta definisi `case1` dan `case2`

In [ ]:
def feature_engineering(df):
    # Salin dataframe agar tidak mengubah data asli secara tidak sengaja
    df_new = df.copy()

    # Klasifikasi kelulusan: jika di bawah 10 maka 'gagal', jika >= 10 maka 'lulus'
    df_new['G1_status'] = df_new['G1'].apply(lambda x: 'gagal' if x < 10 else 'lulus')
    df_new['G2_status'] = df_new['G2'].apply(lambda x: 'gagal' if x < 10 else 'lulus')
    df_new['G3_status'] = df_new['G3'].apply(lambda x: 'gagal' if x < 10 else 'lulus')

    # Definisi case
    # Case 1: G2 dan G3 bernilai 0
    df_new['case1'] = ((df_new['G2'] == 0) & (df_new['G3'] == 0)).astype(int)
    # Case 2: G3 bernilai 0 dan G2 bernilai lebih dari 0
    df_new['case2'] = ((df_new['G3'] == 0) & (df_new['G2'] > 0)).astype(int)

    return df_new

# Terapkan feature engineering pada kedua dataset
df_mat_fe = feature_engineering(df_mat)
df_por_fe = feature_engineering(df_por)

print("Feature Engineering Selesai!")

In [ ]:
# Menampilkan hasil beberapa baris sampel untuk melihat implementasi case1 dan case2
display(df_mat_fe[['G1', 'G2', 'G3', 'G1_status', 'G2_status', 'G3_status', 'case1', 'case2']].head(15))

### 2.2 Label `zero_type`

Dibuat di **dataframe terpisah** (`ZD`) supaya kolom baru tidak ikut terhitung sebagai fitur pada `calculate_association`.
`zero_type`: `A_G2G3` (= case1), `B_G3only` (= case2), `G1zero_anomali` (G1 = 0), `normal`.

In [ ]:
def add_zero_type(df):
    d = df.copy()
    d['zero_type'] = 'normal'
    d.loc[d['case2'] == 1, 'zero_type'] = 'B_G3only'
    d.loc[d['case1'] == 1, 'zero_type'] = 'A_G2G3'
    d.loc[d['G1'] == 0,    'zero_type'] = 'G1zero_anomali'
    return d

ZD = {'Mathematics': add_zero_type(df_mat_fe),
      'Portuguese':  add_zero_type(df_por_fe)}

# Kolom yang bukan fitur (dipakai pada analisis di bagian 4.7)
EXCLUDE_FEATS = ['G1', 'G2', 'G3', 'G1_status', 'G2_status', 'G3_status',
                 'case1', 'case2', 'zero_type']

# Konsistensi: case1 = Type A, case2 = Type B
for lab, d in ZD.items():
    assert (d.case1 == (d.zero_type == 'A_G2G3')).all(), lab
    assert (d.case2 == (d.zero_type == 'B_G3only')).all(), lab

TYPE_COL = {'normal': '#9aa5b1', 'A_G2G3': '#e45756',
            'B_G3only': '#f58518', 'G1zero_anomali': '#54a24b'}

def save_show(fig, name):
    """Simpan gambar ke OUT_DIR lalu tampilkan."""
    fig.savefig(os.path.join(OUT_DIR, name), dpi=DPI, bbox_inches='tight')
    plt.show()

print(pd.DataFrame({k: v.zero_type.value_counts() for k, v in ZD.items()}).fillna(0).astype(int))

# 3. RG1 — Pola Nilai 0 pada G1, G2, G3

### 3.1 Ringkasan jumlah nol

In [ ]:
rows = []
for lab, d in ZD.items():
    z = d[d.zero_type != 'normal']
    rows.append(dict(dataset=lab, n_total=len(d),
                     G1_zero=(d.G1 == 0).sum(), G2_zero=(d.G2 == 0).sum(), G3_zero=(d.G3 == 0).sum(),
                     tipeA=(d.zero_type == 'A_G2G3').sum(),
                     tipeB=(d.zero_type == 'B_G3only').sum(),
                     G1zero=(d.zero_type == 'G1zero_anomali').sum()))
zero_summary = pd.DataFrame(rows)
zero_summary.to_csv(os.path.join(OUT_DIR, 'zero_summary.csv'), index=False)
display(zero_summary)

for lab, d in ZD.items():
    z = d[d.zero_type != 'normal']
    print(f"\n[{lab}] crosstab G2=0 vs G3=0 (hanya siswa bernilai 0)")
    display(pd.crosstab(z.G2 == 0, z.G3 == 0, rownames=['G2=0'], colnames=['G3=0']))
    print(f"[{lab}] G1 & G2 per tipe")
    display(z.groupby('zero_type')[['G1', 'G2']].agg(['count', 'mean', 'min', 'max']).round(2))

### 3.2 Jumlah nol per periode dan per tipe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
mm = zero_summary.melt(id_vars='dataset', value_vars=['G1_zero', 'G2_zero', 'G3_zero'],
                       var_name='period', value_name='count')
mm['period'] = mm.period.str.replace('_zero', '')
sns.barplot(data=mm, x='period', y='count', hue='dataset', ax=axes[0])
for c in axes[0].containers: axes[0].bar_label(c)
axes[0].set_title('Number of zero grades per period')

tt = zero_summary.melt(id_vars='dataset', value_vars=['tipeA', 'tipeB', 'G1zero'],
                       var_name='type', value_name='count')
tt['type'] = tt.type.map({'tipeA': 'Type A\n(G2=G3=0)', 'tipeB': 'Type B\n(G3=0, G2>0)',
                          'G1zero': 'G1=0\n(anomaly)'})
sns.barplot(data=tt, x='type', y='count', hue='dataset', ax=axes[1])
for c in axes[1].containers: axes[1].bar_label(c)
axes[1].set_title('Number of students per zero type')
plt.tight_layout()
save_show(fig, 'figA_bar_jumlah_nol.png')

### 3.3 Sebaran G1 (case1) dan G1 vs G2 (case2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

STYLE = {
    "Portugis":   dict(df=df_por_fe, color="#1f77b4", marker="o", y=1),
    "Matematika": dict(df=df_mat_fe, color="#e8710a", marker="^", y=0),
}
PASS = 10
rng = np.random.default_rng(42)
def jitter(x, s=0.15):
    return x + rng.normal(0, s, len(x))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw={"width_ratios": [1, 1.1]})

# ---------- Case 1: 1 dimensi (hanya G1, karena G2 = 0 untuk semua) ----------
for name, cfg in STYLE.items():
    s = cfg["df"][cfg["df"]["case1"] == 1]
    ax1.scatter(jitter(s.G1, 0.08), cfg["y"] + rng.uniform(-0.18, 0.18, len(s)),
                s=70, c=cfg["color"], marker=cfg["marker"], alpha=.85,
                edgecolor="white", linewidth=.6, zorder=3)
    ax1.axhline(cfg["y"], color="lightgray", lw=1, zorder=1)
    ax1.text(20.3, cfg["y"], f"n={len(s)}", va="center", fontsize=9, color=cfg["color"])

ax1.axvline(PASS, ls="--", c="k", lw=.9)
ax1.axvspan(PASS, 20, color="#2ca02c", alpha=.04)
ax1.set_yticks([0, 1]); ax1.set_yticklabels(["Matematika", "Portugis"])
ax1.set_ylim(-0.6, 1.6); ax1.set_xlim(-0.5, 20)
ax1.set_xlabel("G1")
ax1.set_title("Case 1: G2 = 0 dan G3 = 0 (sebaran G1)", fontsize=11, fontweight="bold")
ax1.grid(axis="x", alpha=.2)

# ---------- Case 2: scatter G1 vs G2 (tetap seperti sebelumnya) ----------
for cfg in STYLE.values():
    bg = cfg["df"][cfg["df"]["case2"] == 0]
    ax2.scatter(jitter(bg.G1), jitter(bg.G2), s=12, c="lightgray", alpha=.35, edgecolor="none", zorder=1)
ax2.scatter([], [], s=12, c="lightgray", label="Siswa lain")
for name, cfg in STYLE.items():
    s = cfg["df"][cfg["df"]["case2"] == 1]
    ax2.scatter(jitter(s.G1), jitter(s.G2), s=60, c=cfg["color"], marker=cfg["marker"],
                alpha=.85, edgecolor="white", linewidth=.6, label=f"{name} (n={len(s)})", zorder=3)
ax2.axvline(PASS, ls="--", c="k", lw=.9); ax2.axhline(PASS, ls="--", c="k", lw=.9)
ax2.axvspan(PASS, 20, color="#2ca02c", alpha=.04)
ax2.set_xlim(-0.5, 20); ax2.set_ylim(-1, 20)
ax2.set_xlabel("G1"); ax2.set_ylabel("G2")
ax2.set_title("Case 2: G3 = 0", fontsize=11, fontweight="bold")
ax2.grid(alpha=.2); ax2.legend(loc="upper left", fontsize=9)

for ax in (ax1, ax2):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)

plt.tight_layout()
plt.show()

### 3.4 Grafik Type A dan Type B (hanya siswa bernilai 0)

Type B hanya memuat G2 > 0 (tanpa titik Type A dan tanpa siswa lain).

In [ ]:
LBL_STYLE = {'Portuguese': dict(c='#1f5fa8', m='o'), 'Mathematics': dict(c='#e8730a', m='^')}
GREEN = '#e6f2e0'

typeA = {k: d[(d.case1 == 1) & (d.G1 > 0)] for k, d in ZD.items()}
typeB = {k: d[d.case2 == 1] for k, d in ZD.items()}

for k in ZD:
    a, b = typeA[k], typeB[k]
    print(f"{k}: Type A n={len(a)}, G1<{PASS_MARK}: {(a.G1 < PASS_MARK).sum()}, G1 range {a.G1.min()}-{a.G1.max()}")
    print(f"{k}: Type B n={len(b)}, G1<{PASS_MARK}: {(b.G1 < PASS_MARK).sum()}, max G2 {b.G2.max()}")

# --- Type A: G1 per dataset ---
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ypos = {'Portuguese': 1, 'Mathematics': 0}
ax.axvspan(PASS_MARK, 20, color=GREEN, zorder=0)
for k, y in ypos.items():
    g = typeA[k]
    ax.axhline(y, color='#999999', lw=1, zorder=1)
    off = [((i % 3) - 1) * 0.07 for i in range(len(g))]        # jitter deterministik kecil
    ax.scatter(g.G1, [y + o for o in off], s=55, c=LBL_STYLE[k]['c'], marker=LBL_STYLE[k]['m'],
               edgecolor='k', linewidth=.7, zorder=3)
    ax.text(20.3, y, f"n={len(g)}", color=LBL_STYLE[k]['c'], va='center', fontsize=9, fontweight='bold')
ax.axvline(PASS_MARK, color='k', ls='--', lw=1.2)
ax.set_yticks(list(ypos.values())); ax.set_yticklabels(list(ypos.keys()))
ax.set_ylim(-.6, 1.6); ax.set_xlim(0, 20); ax.set_xlabel('G1 score')
ax.set_title('Type A (G2 = G3 = 0): G1 score', fontsize=11, fontweight='bold')
ax.legend(handles=[Line2D([], [], color='k', ls='--', label=f'Pass mark ({PASS_MARK})')], loc='upper left', fontsize=8)
ax.grid(False)
plt.tight_layout()
save_show(fig, 'typeA_G1.png')

# --- Type B: G1 vs G2 ---
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.axvspan(PASS_MARK, 20, ymin=PASS_MARK / 20, color=GREEN, zorder=0)
for k in ['Portuguese', 'Mathematics']:
    g = typeB[k]
    ax.scatter(g.G1, g.G2, s=55, c=LBL_STYLE[k]['c'], marker=LBL_STYLE[k]['m'], edgecolor='k',
               linewidth=.7, alpha=.9, zorder=3, label=f"{k} (n={len(g)})")
ax.axvline(PASS_MARK, color='k', ls='--', lw=1.2); ax.axhline(PASS_MARK, color='k', ls='--', lw=1.2)
ax.set_xlim(0, 20); ax.set_ylim(0, 20); ax.set_xlabel('G1 score'); ax.set_ylabel('G2 score')
ax.set_title('Type B (G3 = 0, G2 > 0): G1 vs G2', fontsize=11, fontweight='bold')
h, l = ax.get_legend_handles_labels()
h.append(Line2D([], [], color='k', ls='--', label=f'Pass mark ({PASS_MARK})'))
ax.legend(handles=h, loc='upper left', fontsize=8); ax.grid(alpha=.25)
plt.tight_layout()
save_show(fig, 'typeB_G1_G2.png')

### 3.5 Alur nilai G1 → G2 → G3 (alluvial), hanya siswa bernilai 0

In [ ]:
def zero_state(v):
    return '0' if v == 0 else (f'1-{PASS_MARK-1}' if v < PASS_MARK else f'>={PASS_MARK}')

STATES = ['0', f'1-{PASS_MARK-1}', f'>={PASS_MARK}']
STATE_COL = {'0': '#e45756', f'1-{PASS_MARK-1}': '#f2cf5b', f'>={PASS_MARK}': '#4c78a8'}

def alluvial(ax, z, title):
    cols = ['G1', 'G2', 'G3']
    sub = z[cols]
    st = sub.map(zero_state) if hasattr(sub, 'map') else sub.applymap(zero_state)
    n = len(st); gap, xs, w = 0.03 * n, [0, 1, 2], 0.12
    pos = {}
    for ci, c in enumerate(cols):
        y = 0
        for s in STATES:
            h = (st[c] == s).sum()
            pos[(ci, s)] = [y, h, 0, 0]          # y, tinggi, offset_out, offset_in
            if h:
                ax.add_patch(Rectangle((xs[ci] - w / 2, y), w, h, color=STATE_COL[s], zorder=3))
                ax.text(xs[ci], y + h / 2, f'{s}\n(n={h})', ha='center', va='center', fontsize=7, zorder=4)
            y += h + gap
    for ci in range(2):
        a, b = cols[ci], cols[ci + 1]
        flows = st.groupby([a, b]).size().reset_index(name='n')
        flows['ia'] = flows[a].map(STATES.index); flows['ib'] = flows[b].map(STATES.index)
        for _, r in flows.sort_values(['ia', 'ib']).iterrows():
            pa, pb = pos[(ci, r[a])], pos[(ci + 1, r[b])]
            y0, y1 = pa[0] + pa[2], pb[0] + pb[3]
            pa[2] += r.n; pb[3] += r.n
            x0, x1 = xs[ci] + w / 2, xs[ci + 1] - w / 2; xm = (x0 + x1) / 2
            verts = [(x0, y0), (xm, y0), (xm, y1), (x1, y1), (x1, y1 + r.n), (xm, y1 + r.n),
                     (xm, y0 + r.n), (x0, y0 + r.n), (x0, y0)]
            codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.LINETO,
                     Path.CURVE4, Path.CURVE4, Path.CURVE4, Path.CLOSEPOLY]
            ax.add_patch(PathPatch(Path(verts, codes), fc=STATE_COL[r[a]], ec='none', alpha=.45, zorder=2))
    ax.set_xlim(-.3, 2.3)
    ax.set_ylim(-gap, max(sum(p[1] for k, p in pos.items() if k[0] == c) + 3 * gap for c in range(3)))
    ax.set_xticks(xs); ax.set_xticklabels(cols); ax.set_yticks([]); ax.grid(False)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, (lab, d) in zip(axes, ZD.items()):
    z = d[d.zero_type != 'normal']
    alluvial(ax, z, f'{lab}: grade flow of students with a zero (n={len(z)})')
plt.tight_layout()
save_show(fig, 'figB_alluvial_G1_G2_G3.png')

### 3.6 Lintasan nilai per siswa (slope chart, opsional)

In [ ]:
rng_slope = np.random.default_rng(SEED)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, (lab, d) in zip(axes, ZD.items()):
    z = d[d.zero_type != 'normal']
    for t_, g in z.groupby('zero_type'):
        for _, r in g.iterrows():
            jit = rng_slope.uniform(-.25, .25)          # jitter vertikal kecil
            ax.plot([0, 1, 2], np.array([r.G1, r.G2, r.G3]) + jit, color=TYPE_COL[t_],
                    alpha=.6, lw=1.2, marker='o', ms=3)
    ax.axhline(PASS_MARK, ls='--', c='grey', lw=1)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['G1', 'G2', 'G3'])
    ax.set_title(f'{lab}: grade trajectory per student')
axes[0].set_ylabel('grade (small jitter for readability)')
axes[0].legend(handles=[Line2D([], [], color=TYPE_COL[t], label=t) for t in ['A_G2G3', 'B_G3only', 'G1zero_anomali']],
               fontsize=8, loc='upper right')
plt.tight_layout()
save_show(fig, 'figC_slope_per_siswa.png')

# 4. RG2 — Fitur yang Berasosiasi dengan Nilai 0

### 4.1 Asosiasi fitur per case (Chi-square / Cramér's V dan Mann-Whitney)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

def calculate_association(df, target_col):
    # Pisahkan fitur demografis/akademis sebelum G1, G2, G3
    exclude_cols = ['G1', 'G2', 'G3', 'G1_status', 'G2_status', 'G3_status', 'case1', 'case2']
    features = [c for c in df.columns if c not in exclude_cols]

    results = []

    # Cek jika target hanya memiliki 1 kelas unik (misal tidak ada kasus sama sekali)
    if df[target_col].nunique() < 2:
        return pd.DataFrame(columns=['Feature', 'Type', 'Stat_Value', 'p-value'])

    for col in features:
        # Jika kategorikal (object)
        if df[col].dtype == 'object':
            contingency_table = pd.crosstab(df[col], df[target_col])
            # Chi-square test
            try:
                chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)
                # Hitung Cramer's V
                n = contingency_table.sum().sum()
                min_dim = min(contingency_table.shape) - 1
                cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
                results.append({
                    'Feature': col,
                    'Type': 'Categorical (Cramer\'s V)',
                    'Association_Strength': cramers_v,
                    'p-value': p_val
                })
            except Exception:
                pass
        # Jika numerik
        else:
            # Bandingkan nilai fitur antara kelompok target=1 dan target=0
            group1 = df[df[target_col] == 1][col]
            group0 = df[df[target_col] == 0][col]
            if len(group1) > 0 and len(group0) > 0:
                try:
                    stat, p_val = stats.mannwhitneyu(group1, group0, alternative='two-sided')
                    # Gunakan korelasi rank biserial sederhana sebagai estimasi kekuatan asosiasi
                    n1, n2 = len(group1), len(group0)
                    rank_biserial = 1 - (2 * stat) / (n1 * n2)
                    results.append({
                        'Feature': col,
                        'Type': 'Numerical (Mann-Whitney)',
                        'Association_Strength': abs(rank_biserial),
                        'p-value': p_val
                    })
                except Exception:
                    pass

    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df = res_df.sort_values(by='p-value').reset_index(drop=True)
    return res_df

# Analisis untuk Matematika
print("=== FITUR PALING SIGNIFIKAN UNTUK CASE 1 (G2 & G3 = 0) - MATEMATIKA ===")
mat_case1 = calculate_association(df_mat_fe, 'case1')
display(mat_case1.head(10))

print("\n=== FITUR PALING SIGNIFIKAN UNTUK CASE 2 (G3 = 0) - MATEMATIKA ===")
mat_case2 = calculate_association(df_mat_fe, 'case2')
display(mat_case2.head(10))

### 4.2 Fitur Bahasa Portugis

In [ ]:
# Analisis untuk Portugis
print("=== FITUR PALING SIGNIFIKAN UNTUK CASE 1 (G2 & G3 = 0) - PORTUGIS ===")
por_case1 = calculate_association(df_por_fe, 'case1')
display(por_case1.head(15))

print("\n=== FITUR PALING SIGNIFIKAN UNTUK CASE 2 (G3 = 0) - PORTUGIS ===")
por_case2 = calculate_association(df_por_fe, 'case2')
display(por_case2.head(15))

### 4.3 Grid plot fitur teratas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_top_features(df_res, title, ax):
    # Ambil 10 fitur teratas
    top_feats = df_res.head(10).copy()

    # Hitung -log10(p-value) untuk visualisasi tingkat signifikansi yang intuitif
    # Tambahkan epsilon kecil untuk menghindari log(0)
    top_feats['log_p'] = -np.log10(top_feats['p-value'] + 1e-20)

    # Plot bar horizontal
    sns.barplot(
        x='log_p',
        y='Feature',
        data=top_feats,
        palette='viridis',
        ax=ax,
        hue='Type',
        legend=True
    )

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('-log10(p-value) [Semakin tinggi = Semakin Signifikan]', fontsize=10)
    ax.set_ylabel('Fitur', fontsize=10)
    ax.axvline(x=-np.log10(0.05), color='red', linestyle='--', alpha=0.7, label='Batas Signifikan (p=0.05)')
    ax.legend(loc='lower right', fontsize=8)

# Siapkan grid plot 2x2 untuk menampilkan semua skenario
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Matematika - Case 1
plot_top_features(mat_case1, "Matematika - Fitur Penentu Case 1 (G2 & G3 = 0)", axes[0, 0])

# 2. Matematika - Case 2
plot_top_features(mat_case2, "Matematika - Fitur Penentu Case 2 (G3 = 0)", axes[0, 1])

# 3. Portugis - Case 1
plot_top_features(por_case1, "Portugis - Fitur Penentu Case 1 (G2 & G3 = 0)", axes[1, 0])

# 4. Portugis - Case 2
plot_top_features(por_case2, "Portugis - Fitur Penentu Case 2 (G3 = 0)", axes[1, 1])

plt.tight_layout()
plt.show()

### 4.4 Panel per case

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def add_direction(res, df, target):
    """Tambah kolom arah (↑/↓) untuk fitur numerik."""
    r = res.copy()
    arrows = []
    for f, t in zip(r["Feature"], r["Type"]):
        if t.startswith("Numerical"):
            a, b = df.loc[df[target] == 1, f].mean(), df.loc[df[target] == 0, f].mean()
            arrows.append("↑" if a > b else "↓")
        else:
            arrows.append("")
    r["arah"] = arrows
    return r

def fmt_p(p):
    s = f"{p:.4f}"
    return "<0.0001" if s == "0.0000" else s

C_SIG, C_NS = "#0b2a5b", "#cfd3d8"

def plot_single_panel(res, title, top=10):
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    d = res.head(top).iloc[::-1]
    x = -np.log10(d["p-value"].values)
    y = np.arange(len(d))
    thr = -np.log10(0.05)

    ax.axvspan(0, thr, color="#eef0f2", zorder=0)
    for xi, yi, (_, row) in zip(x, y, d.iterrows()):
        sig = row["p-value"] < 0.05
        ax.barh(yi, xi, height=0.72, color=C_SIG if sig else C_NS,
                edgecolor="black" if sig else "none", linewidth=1.2,
                hatch="///" if row["Type"].startswith("Categorical") else None, zorder=2)
        ax.text(max(xi, thr) + x.max() * 0.015, yi, fmt_p(row["p-value"]),
                va="center", fontsize=10.5, fontweight="bold" if sig else "normal",
                color="black" if sig else "#777")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{f} {a}" for f, a in zip(d["Feature"], d["arah"])], fontsize=12)
    for lbl, (_, row) in zip(ax.get_yticklabels(), d.iterrows()):
        sig = row["p-value"] < 0.05
        lbl.set_fontweight("bold" if sig else "normal")
        lbl.set_color("black" if sig else "#888")
    ax.axvline(thr, color="#d62728", ls="--", lw=2.4, zorder=3)
    ax.text(thr, len(d) - 0.35, " p = 0.05", color="#d62728", fontsize=10, fontweight="bold", va="bottom")
    ax.set_xlim(0, x.max() * 1.22); ax.set_ylim(-0.7, len(d) - 0.1)
    ax.set_xlabel("−log10(p-value)", fontsize=11, fontweight="bold")
    ax.set_title(title, fontsize=13, fontweight="bold", loc="left", pad=12)
    ax.grid(axis="x", alpha=.3, zorder=1)
    ax.tick_params(axis="x", labelsize=10)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"):
        ax.spines[sp].set_linewidth(1.5)

    legend_handles = [
        Patch(facecolor=C_SIG, edgecolor="black", label="Signifikan (p < 0.05)"),
        Patch(facecolor=C_NS, label="Tidak signifikan"),
        Patch(facecolor="white", edgecolor="black", hatch="///", label="Fitur kategorikal (Chi-square)"),
        Patch(facecolor="white", edgecolor="black", label="Fitur numerik (Mann-Whitney)")
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=9, frameon=True, edgecolor="black")
    plt.tight_layout()
    plt.show()

mat1 = add_direction(mat_case1, df_mat_fe, "case1")
mat2 = add_direction(mat_case2, df_mat_fe, "case2")
por1 = add_direction(por_case1, df_por_fe, "case1")
por2 = add_direction(por_case2, df_por_fe, "case2")

# Menampilkan setiap panel sebagai visualisasi mandiri terpisah
plot_single_panel(mat1,"" )
plot_single_panel(mat2, "")
plot_single_panel(por1, "")
plot_single_panel(por2, "")

### 4.5 Absences pada siswa bernilai 0

Absences = 0 pada semua siswa nol adalah **pola pencatatan**, bukan "faktor penyebab".

In [ ]:
abs_rows = []
for lab, d in ZD.items():
    for name, mask in [('Type A (case1)', d.case1 == 1), ('Type B (case2)', d.case2 == 1),
                       ('All zeros (G3=0)', d.G3 == 0), ('Students with G3>0', d.G3 > 0)]:
        s = d[mask]
        abs_rows.append(dict(dataset=lab, group=name, n=len(s), mean_absences=round(s.absences.mean(), 2),
                             absences_zero=f"{(s.absences == 0).sum()}/{len(s)}"))
absences_table = pd.DataFrame(abs_rows)
absences_table.to_csv(os.path.join(OUT_DIR, 'absences_by_group.csv'), index=False)
display(absences_table)

### 4.6 Koreksi multiple testing (FDR)

`calculate_association` menguji puluhan fitur sekaligus, sehingga beberapa p < 0.05 bisa muncul karena kebetulan.
Kolom `q_fdr` (Benjamini–Hochberg) ditambahkan sebagai koreksi.

In [ ]:
def bh_fdr(p):
    """Benjamini-Hochberg q-value."""
    p = np.asarray(p, float); o = np.argsort(p); m = len(p)
    r = p[o] * m / (np.arange(m) + 1)
    q = np.empty(m); q[o] = np.minimum.accumulate(r[::-1])[::-1].clip(max=1)
    return q

def add_fdr(res):
    r = res.copy()
    r['q_fdr'] = bh_fdr(r['p-value'].values)
    r['sig_fdr'] = r['q_fdr'] < 0.05
    return r

fdr_tables = {
    'Mathematics - case1 (Type A)': add_fdr(mat_case1),
    'Mathematics - case2 (Type B)': add_fdr(mat_case2),
    'Portuguese - case1 (Type A)':  add_fdr(por_case1),
    'Portuguese - case2 (Type B)':  add_fdr(por_case2),
}
for name, t in fdr_tables.items():
    print(f"\n=== {name}: fitur yang tetap signifikan setelah FDR (q < 0.05) ===")
    keep = t[t.sig_fdr][['Feature', 'Type', 'Association_Strength', 'p-value', 'q_fdr']]
    display(keep if len(keep) else pd.DataFrame({'info': ['tidak ada']}))

### 4.7 Type A / Type B dibandingkan dengan siswa G3 > 0

Pada analisis 4.1, pembandingnya adalah **semua siswa lain** (termasuk tipe nol lainnya), sehingga efek tipe nol lain ikut mengencerkan hasil.
Di sini pembandingnya hanya **siswa dengan G3 > 0**; 2×2 kategorikal memakai Fisher exact, ditambah koreksi FDR.
Portuguese hanya ditampilkan sebagai tabel karena Type A (n = 7) dan Type B (n = 8) terlalu kecil untuk kesimpulan per fitur.

In [ ]:
def compare_features(d, group_mask, top_n=9):
    """Kelompok (Type A / B) vs siswa G3>0, dengan koreksi FDR."""
    ref = d.G3 > 0
    sub = d[group_mask | ref]; gg = group_mask[group_mask | ref]
    feats = [c for c in d.columns if c not in EXCLUDE_FEATS]
    rows = []
    for f in feats:
        x = sub[f]
        if pd.api.types.is_numeric_dtype(x):
            p = stats.mannwhitneyu(x[gg], x[~gg]).pvalue
            test = 'Mann-Whitney'; arrow = '↑' if x[gg].mean() > x[~gg].mean() else '↓'
        else:
            ct = pd.crosstab(x, gg)
            if ct.shape == (2, 2):
                p = stats.fisher_exact(ct)[1]; test = 'Fisher exact'
            else:
                p = stats.chi2_contingency(ct, correction=False)[1] if ct.shape[0] > 1 else 1.0
                test = 'Chi-square'
            arrow = ''
        rows.append(dict(feature=f + (f' {arrow}' if arrow else ''), p=p, test=test))
    r = pd.DataFrame(rows); r['q'] = bh_fdr(r.p.values)
    return r.sort_values('p').head(top_n).iloc[::-1], len(gg), int(gg.sum())

FEAT_DATASET = 'Mathematics'          # <-- dataset untuk grafik (disarankan Mathematics)
d_sel = ZD[FEAT_DATASET]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, grp) in zip(axes, [('Type A (G2 = G3 = 0)', d_sel.case1 == 1),
                                  ('Type B (G3 = 0, G2 > 0)', d_sel.case2 == 1)]):
    r, n_all, n_g = compare_features(d_sel, grp)
    colors = ['#1f3b73' if q < 0.05 else '#c9cdd2' for q in r.q]
    xv = -np.log10(r.p.clip(lower=1e-6))                  # batang dibatasi agar terbaca
    ax.barh(r.feature, xv, color=colors)
    for y, (xx, q) in enumerate(zip(xv, r.q)):
        ax.text(xx + .05, y, f'q = {q:.3f}', va='center', fontsize=8)
    ax.axvline(-np.log10(0.05), color='red', ls='--', lw=1)
    ax.set_title(f'{name}\nn = {n_g} vs {n_all - n_g} students with G3 > 0', fontsize=10, fontweight='bold')
    ax.set_xlabel('−log10(p-value)'); ax.set_xlim(0, max(xv) * 1.25)
    display(r.iloc[::-1].round(4).assign(group=name))
axes[1].legend(handles=[Patch(color='#1f3b73', label='Significant after FDR (q < 0.05)'),
                        Patch(color='#c9cdd2', label='Not significant after FDR')],
               loc='lower right', fontsize=8)
fig.suptitle(f'{FEAT_DATASET}: features associated with zero grades (↑/↓ = zero group higher/lower)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
save_show(fig, f'features_typeA_typeB_{FEAT_DATASET[:3].lower()}.png')

In [ ]:
# Portuguese: tabel saja (n kecil, hasil per fitur tidak stabil)
d_p = ZD['Portuguese']
for name, grp in [('Type A', d_p.case1 == 1), ('Type B', d_p.case2 == 1)]:
    r, n_all, n_g = compare_features(d_p, grp, top_n=6)
    print(f"Portuguese {name}: n = {n_g} vs {n_all - n_g} (G3 > 0) -- interpretasi hati-hati, sampel kecil")
    display(r.iloc[::-1].round(4))

# 5. RG3 — Sensitivitas Korelasi Absences vs G3 terhadap Nilai 0

**Pengecualian:** bagian ini memakai **semua siswa** (nol dan bukan nol) untuk membandingkan skenario.

| Skenario | Perlakuan |
|---|---|
| S0 | Semua siswa |
| S1 | Keluarkan Type A (`case1`) |
| S2 | Keluarkan Type B (`case2`) |
| S3 | Keluarkan Type A dan B (semua G3 = 0) |

Kasus tunggal G1 = 0 (Portuguese, G2 = G3 = 11) tidak dikeluarkan di S3 karena G3-nya bukan nol (hasil tidak berubah bila dikeluarkan: r ≈ −0.196).

In [ ]:
def corr_ci(x, y, method='pearson', n_boot=N_BOOT, seed=SEED):
    """Korelasi + 95% bootstrap CI (seed tetap per panggilan -> CI konsisten antar sel)."""
    rng_b = np.random.default_rng(seed)
    x, y = np.asarray(x, float), np.asarray(y, float)
    r = pd.Series(x).corr(pd.Series(y), method=method)
    boots = []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(x), len(x))
        boots.append(pd.Series(x[i]).corr(pd.Series(y[i]), method=method))
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return r, lo, hi

SCEN = {'S0': 'All students', 'S1': 'Exclude Type A',
        'S2': 'Exclude Type B', 'S3': 'Exclude Type A and B'}

def scenario_subsets(d):
    return {'S0': d,
            'S1': d[d.case1 == 0],
            'S2': d[d.case2 == 0],
            'S3': d[(d.case1 == 0) & (d.case2 == 0)]}

rows = []
for lab, d in ZD.items():
    for sk, sub in scenario_subsets(d).items():
        r, lo, hi = corr_ci(sub.absences, sub.G3)
        rho, p_rho = stats.spearmanr(sub.absences, sub.G3)
        rows.append(dict(dataset=lab, scenario=sk, treatment=SCEN[sk], n=len(sub), removed=len(d) - len(sub),
                         mean_absences=round(sub.absences.mean(), 2), mean_G3=round(sub.G3.mean(), 2),
                         r=round(r, 3), ci_low=round(lo, 3), ci_high=round(hi, 3),
                         spearman_rho=round(rho, 3), p_spearman=round(p_rho, 4)))
sens = pd.DataFrame(rows)
sens.to_csv(os.path.join(OUT_DIR, 'sensitivity_results.csv'), index=False)
display(sens)

### 5.1 Bar chart korelasi per skenario (dengan CI 95%)

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.8))
w = 0.2
palette = ['#9aa5b1', '#7fa8d6', '#4c78a8', '#1f3b73']
labs = list(ZD)
for j, sk in enumerate(SCEN):
    sub = sens[sens.scenario == sk].set_index('dataset').loc[labs]
    x = np.arange(len(labs)) + (j - 1.5) * w
    err = [sub.r - sub.ci_low, sub.ci_high - sub.r]
    ax.bar(x, sub.r, w, color=palette[j], edgecolor='k', linewidth=.5, yerr=err, capsize=3,
           label=f'{sk}: {SCEN[sk]}')
    for xi, r_, lo_ in zip(x, sub.r, sub.ci_low):
        ax.text(xi, lo_ - 0.015, f'{r_:+.2f}', ha='center', va='top', fontsize=8)
ax.axhline(0, color='k', lw=1)
ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs)
ax.set_ylabel('Pearson r (absences vs G3)'); ax.set_ylim(-0.4, 0.2)
ax.set_title('Absences–G3 correlation across scenarios', fontsize=11, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.text(0.01, 0.01, 'Error bars: 95% bootstrap CI', transform=ax.transAxes, fontsize=7, color='#555555')
ax.grid(axis='y', alpha=.25)
plt.tight_layout()
save_show(fig, 'bar_S0_S3_four_scenarios.png')

### 5.2 Scatter per skenario *(opsional, ubah `PLOT_SCATTER` menjadi True)*

In [ ]:
PLOT_SCATTER = False      # <-- ubah ke True untuk membuat 4 gambar scatter

if PLOT_SCATTER:
    for sk, name in SCEN.items():
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
        for ax, lab in zip(axes, ZD):
            sub = scenario_subsets(ZD[lab])[sk]
            for g, gl in [('normal', 'G3 > 0'), ('A_G2G3', 'Type A'), ('B_G3only', 'Type B')]:
                s = sub[sub.zero_type == g] if g != 'normal' else sub[~sub.zero_type.isin(['A_G2G3', 'B_G3only'])]
                if len(s):
                    ax.scatter(s.absences, s.G3, s=16, alpha=.6, c=TYPE_COL[g], label=gl)
            sl, ic, *_ = stats.linregress(sub.absences, sub.G3)
            xs = np.array([0, sub.absences.max()])
            ax.plot(xs, ic + sl * xs, color='k', lw=2, ls='--')
            r, lo, hi = corr_ci(sub.absences, sub.G3)
            ax.set_title(f'{lab} (n={len(sub)})', fontsize=10, fontweight='bold')
            ax.text(0.97, 0.97, f'r = {r:+.3f}\n95% CI [{lo:+.3f}, {hi:+.3f}]', transform=ax.transAxes,
                    ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', fc='white', ec='#bbbbbb'))
            ax.set_xlabel('Absences'); ax.grid(alpha=.25)
        axes[0].set_ylabel('G3'); axes[0].legend(fontsize=8, loc='lower right')
        fig.suptitle(f'{sk}: {name}', fontsize=12, fontweight='bold')
        plt.tight_layout()
        save_show(fig, f'scatter_{sk}.png')

# 6. Pengecekan Angka untuk Slide

Sel ini memastikan angka yang dipakai di slide sama dengan hasil kode.

In [ ]:
checks = {
 'Math  Type A n=13':        (ZD['Mathematics'].case1.sum(), 13),
 'Math  Type B n=25':        (ZD['Mathematics'].case2.sum(), 25),
 'Math  G3=0 total=38':      ((ZD['Mathematics'].G3 == 0).sum(), 38),
 'Math  G2=0 -> G3=0 (13)':  (((ZD['Mathematics'].G2 == 0) & (ZD['Mathematics'].G3 == 0)).sum(), (ZD['Mathematics'].G2 == 0).sum()),
 'Por   Type A n=7':         (ZD['Portuguese'].case1.sum(), 7),
 'Por   Type B n=8':         (ZD['Portuguese'].case2.sum(), 8),
 'Por   G3=0 total=15':      ((ZD['Portuguese'].G3 == 0).sum(), 15),
 'Por   G1=0 cases=1':       ((ZD['Portuguese'].G1 == 0).sum(), 1),
 'All G3=0 have absences=0': (sum(((d.G3 == 0) & (d.absences == 0)).sum() for d in ZD.values()), 53),
 'Math  S0 r (+0.034)':      (round(sens.query("dataset=='Mathematics' and scenario=='S0'").r.iloc[0], 3), 0.034),
 'Math  S3 r (-0.213)':      (round(sens.query("dataset=='Mathematics' and scenario=='S3'").r.iloc[0], 3), -0.213),
 'Por   S0 r (-0.091)':      (round(sens.query("dataset=='Portuguese' and scenario=='S0'").r.iloc[0], 3), -0.091),
 'Por   S3 r (-0.196)':      (round(sens.query("dataset=='Portuguese' and scenario=='S3'").r.iloc[0], 3), -0.196),
}
ok = True
for k, (got, exp) in checks.items():
    status = 'OK ' if got == exp else 'BEDA'
    ok &= (got == exp)
    print(f'{status} {k:28s} hasil={got}  harapan={exp}')
print('\nSemua angka sesuai slide.' if ok else '\nAda angka yang berbeda, periksa dataset/versi.')
print('Output tersimpan di:', os.path.abspath(OUT_DIR))